# LAB7 Assignment
> The document description are designed by JIa Yanhong in 2022. Oct. 20th
------

In [1]:
# 这一单元负责把后面会用到的库一次性导入进来
# pandas: 读取 CSV
# numpy: 数组处理
# torch: PyTorch 主框架
# train_test_split: 切分训练集和测试集
# DataLoader / TensorDataset: 给 MLP 训练时做批量加载
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# 为了结果尽量可复现，固定随机种子
torch.manual_seed(42)
np.random.seed(42)

# 自动选择可用设备：优先 CUDA，其次 MPS，最后 CPU
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

Using device: mps


## LAB Assignment
### Exercise 1 logistic regression
This exercise uses dataset digit01.csv , which has 13 columns, and the last column is the dependent variable. 

This part requires you to implement a `logistic regression` using the pytorch framework (defining a logistic regression class that inherits `nn.module`). To test your model, we provide a dataset `digit01.csv` which is in the **datasets folder**. This dataset requires you to divide the training set and the test set by yourself, and it is recommended that 80% of the training set and 20% of the test set be used.

+ load datasets

In [2]:
########### Write Your Code Here ###########

df = pd.read_csv("datasets/digit01.csv", header=None)

print("Dataset shape:", df.shape)
print(df.head())

############################################

Dataset shape: (64, 13)
   0   1   2   3   4   5   6   7   8   9   10  11  12
0   1   1   1   1   0   1   1   0   1   1   1   1   0
1   0   1   1   1   0   1   1   0   1   1   1   1   0
2   1   1   0   1   0   1   1   0   1   1   1   1   0
3   1   1   1   1   0   1   1   0   1   1   1   0   0
4   1   1   1   1   0   1   1   0   1   0   1   1   0


+ Splitting dataset into 80% Training and 20% Testing Data:

In [4]:
########### Write Your Code Here ###########

# [:, :-1]: select all rows, and all columns except the last one (features)
X = df.iloc[:, :-1].values.astype(np.float32)

# [:, -1]: select all rows, and only the last column (labels)
y = df.iloc[:, -1].values.astype(np.int64)

# split the dataset into training and testing sets
# with 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size = 0.8,
    random_state = 42,
    stratify = y
)

# 转成 PyTorch Tensor
# X 用 float32，方便喂给线性层
# y_train 后面会用于 BCEWithLogitsLoss，所以要变成 float 并变成二维列向量
X_train = torch.from_numpy(X_train).to(device)
X_test = torch.from_numpy(X_test).to(device)
y_train = torch.from_numpy(y_train).float().unsqueeze(1).to(device)

# y_test 先保留为 numpy 数组，后面 classification_report 会直接用它
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

############################################

X_train shape: torch.Size([51, 12])
X_test shape: torch.Size([13, 12])
y_train shape: torch.Size([51, 1])
y_test shape: (13,)


+ Define a LogisticRegression subclass of nn. Module

In [13]:
# Define a LogisticRegression subclass of nn. Module.
########### Write Your Code Here ###########

class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # the first layer: input_dim (input) -> 1 (output)
        # output: logits, but not probabilities, so no activation function here
        self.linear = nn.Linear(input_dim, 1) 

    def forward(self, x):
        # x: [batch_size, input_dim]
        # output: [batch_size, 1]
        return self.linear(x)

############################################

+ Create the model

In [14]:

########### Write Your Code Here ###########

# input_dim is the number of features, which is the number of columns in X_train
input_dim = X_train.shape[1]

# instantiate the model and move it to the device
model = LogisticRegression(input_dim).to(device)

print(model)

############################################

LogisticRegression(
  (linear): Linear(in_features=12, out_features=1, bias=True)
)


 + Loss function

In [15]:
########### Write Your Code Here ###########

# BCEWithLogitsLoss combines a Sigmoid layer and the BCELoss in one single class.
# This version is more numerically stable than using a plain Sigmoid followed by a BCELoss.
criterion = nn.BCEWithLogitsLoss()

############################################

+ The optimizer

In [16]:
########### Write Your Code Here ###########

# Adam is an adaptive learning rate optimization algorithm
# that's been designed specifically for training deep neural networks.
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

############################################

+ training Model

In [17]:
########### Write Your Code Here ###########

num_epochs = 300 # number of times to loop over the entire training dataset

for epoch in range(num_epochs):
    model.train() # set the model to training mode
    optimizer.zero_grad() # clear the gradients before each training step

    logits = model(X_train) # forward pass: compute the logits
    loss = criterion(logits, y_train) # compute the loss

    loss.backward() # backward pass: compute the gradients

    optimizer.step() # update the parameters using the optimizer

    if (epoch + 1) % 50 == 0: # print the loss every 50 epochs
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

############################################

Epoch [50/300], Loss: 0.2785
Epoch [100/300], Loss: 0.1512
Epoch [150/300], Loss: 0.1017
Epoch [200/300], Loss: 0.0760
Epoch [250/300], Loss: 0.0603
Epoch [300/300], Loss: 0.0496



+ Model Performance


In [21]:
########### Write Your Code Here ###########

model.eval() # set the model to evaluation mode

with torch.no_grad(): # disable gradient calculation for evaluation
    test_logits = model(X_test) # compute the logits for the test set
    test_probs = torch.sigmoid(test_logits) # apply sigmoid to get probabilities

    y_predicted_cls = (test_probs >= 0.5).long().cpu().numpy().ravel() # convert probabilities to binary predictions (0 or 1)

    test_accuracy = (y_predicted_cls == y_test).mean() # compute accuracy
    
    print(f"Test Accuracy: {test_accuracy:.4f}") # print the test accuracy

############################################

Test Accuracy: 1.0000


In [22]:
#classification report
from sklearn.metrics import classification_report
print(classification_report(y_test, y_predicted_cls))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       1.00      1.00      1.00         6

    accuracy                           1.00        13
   macro avg       1.00      1.00      1.00        13
weighted avg       1.00      1.00      1.00        13



### Exercise 2  Handwriting recognition with MLP

Like last week's lab , your task in this section is also about recognizing handwritten digits, but you are required to use MLP to complete the exercise. It is recommended that you define an MLP class, which is a subclass of `nn.module`.


For this exercise we use the `minist` dataset.

+ load datasets

In [23]:
########### Write Your Code Here ###########

# 这一部分不依赖 torchvision，直接读取 MNIST 的原始 ubyte 文件
# 这样即使没有额外下载工具，也能离线完成作业
import struct

def read_image(file_name):
    # MNIST 图像文件格式：
    # 前 16 字节是文件头，包含魔数、图片数、行数、列数
    # 后面全部是像素值
    with open(file_name, "rb") as file_handle:
        file_content = file_handle.read()

    head = struct.unpack_from(">IIII", file_content, 0)
    offset = struct.calcsize(">IIII")

    img_num = head[1]
    width = head[2]
    height = head[3]

    bits = img_num * width * height
    bits_string = ">" + str(bits) + "B"
    imgs = struct.unpack_from(bits_string, file_content, offset)

    # reshape 成 [样本数, 高, 宽]
    return np.array(imgs).reshape((img_num, width, height))

def read_label(file_name):
    # MNIST 标签文件格式：
    # 前 8 字节是文件头，后面是每个样本对应的标签
    with open(file_name, "rb") as file_handle:
        file_content = file_handle.read()

    head = struct.unpack_from(">II", file_content, 0)
    offset = struct.calcsize(">II")

    label_num = head[1]
    bits_string = ">" + str(label_num) + "B"
    labels = struct.unpack_from(bits_string, file_content, offset)

    return np.array(labels)

# 拼出 MNIST 原始文件路径
train_image_path = os.path.join("datasets", "MNIST", "raw", "train-images-idx3-ubyte")
train_label_path = os.path.join("datasets", "MNIST", "raw", "train-labels-idx1-ubyte")
test_image_path = os.path.join("datasets", "MNIST", "raw", "t10k-images-idx3-ubyte")
test_label_path = os.path.join("datasets", "MNIST", "raw", "t10k-labels-idx1-ubyte")

# 读取原始图像和标签
train_x = read_image(train_image_path)
train_y = read_label(train_label_path)
test_x = read_image(test_image_path)
test_y = read_label(test_label_path)

# 归一化到 [0, 1]
train_x = train_x.astype(np.float32) / 255.0
test_x = test_x.astype(np.float32) / 255.0

# 标签转成 int64，CrossEntropyLoss 要求目标是整数类别编号
train_y = train_y.astype(np.int64)
test_y = test_y.astype(np.int64)

print("train_x shape:", train_x.shape)
print("test_x shape:", test_x.shape)
print("train_y shape:", train_y.shape)
print("test_y shape:", test_y.shape)

############################################                      

train_x shape: (60000, 28, 28)
test_x shape: (10000, 28, 28)
train_y shape: (60000,)
test_y shape: (10000,)


+ Define a MLP subclass of nn. Module

In [24]:
########### Write Your Code Here ###########

class MLP(nn.Module):
    # input_dim: grayscale image: 28x28=784
    # hidden1: the number of neurons in the first hidden layer
    # hidden2: the number of neurons in the second hidden layer
    # num_classes: the number of output classes, which is 10 for MNIST
    def __init__(self, input_dim = 28 * 28, hidden1 = 256, hidden2 = 128, num_classes = 10):
        super().__init__()
        self.flatten = nn.Flatten() # flatten the input image into a vector

        self.fc1 = nn.Linear(input_dim, hidden1) # first fully connected layer
        self.relu1 = nn.ReLU() # activation function for the first hidden layer

        self.fc2 = nn.Linear(hidden1, hidden2) # second fully connected layer
        self.relu2 = nn.ReLU() # activation function for the second hidden layer

        self.fc3 = nn.Linear(hidden2, num_classes) # output layer

    # x: [batch_size, 1, 28, 28]
    # output: [batch_size, num_classes]
    def forward(self, x):
        x = self.flatten(x) # flatten the input image into a vector

        x = self.relu1(self.fc1(x)) # pass through the first hidden layer and activation

        x = self.relu2(self.fc2(x)) # pass through the second hidden layer and activation

        x = self.fc3(x) # pass through the output layer (logits)

        return x

############################################

+ Create the model

In [25]:
########### Write Your Code Here ###########

# Convert numpy arrays to PyTorch tensors and move to device
X_train_tensor = torch.from_numpy(train_x).to(device)
X_test_tensor = torch.from_numpy(test_x).to(device)
y_train_tensor = torch.from_numpy(train_y).to(device)
y_test_tensor = torch.from_numpy(test_y).to(device)

# construct TensorDataset for training and testing sets
# so that we can use DataLoader to load them in batches during training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# construct DataLoader for training and testing sets
# batch_size: the number of samples per batch to load
# shuffle: whether to shuffle the data at every epoch (True for training, False for testing)
train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 128, shuffle = False)

# instantiate the MLP model and move it to the device
mlp_model = MLP().to(device)
print(mlp_model)

############################################

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)


 + Loss function

In [26]:
########### Write Your Code Here ###########

# CrossEntropyLoss combines a LogSoftmax layer and the NLLLoss in one single class.
# This version is more numerically stable than using a plain Softmax followed by a NLLLoss.
# CrossEntropyLoss expects the model output to be raw logits (not probabilities)
# and the target to be class indices (not one-hot).
mlp_criterion = nn.CrossEntropyLoss()

############################################

+ The optimizer

In [27]:
########### Write Your Code Here ###########

# Adam is an adaptive learning rate optimization algorithm
# that's been designed specifically for training deep neural networks.
mlp_optimizer = torch.optim.Adam(mlp_model.parameters(), lr = 0.001)

############################################

+ training Model

In [29]:

########### Write Your Code Here ###########

mlp_epochs = 10 # number of times to loop over the entire training dataset

for epoch in range(mlp_epochs):
    mlp_model.train() # set the model to training mode
    running_loss = 0.0 # variable to accumulate the loss over batches

    for batch_x, batch_y in train_loader:
        # move the batch data to the device
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        mlp_optimizer.zero_grad() # clear the gradients before each training step

        logits = mlp_model(batch_x) # forward pass: compute the logits

        loss = mlp_criterion(logits, batch_y) # compute the loss

        loss.backward() # backward pass: compute the gradients

        mlp_optimizer.step() # update the parameters using the optimizer

        running_loss += loss.item() # accumulate the loss for this batch
    
    avg_loss = running_loss / len(train_loader) # compute the average loss for this epoch
    print(f"Epoch [{epoch + 1}/{mlp_epochs}], Loss: {avg_loss:.4f}") # print the average loss for this epoch

############################################

Epoch [1/10], Loss: 0.3441
Epoch [2/10], Loss: 0.1348
Epoch [3/10], Loss: 0.0880
Epoch [4/10], Loss: 0.0648
Epoch [5/10], Loss: 0.0495
Epoch [6/10], Loss: 0.0369
Epoch [7/10], Loss: 0.0304
Epoch [8/10], Loss: 0.0228
Epoch [9/10], Loss: 0.0184
Epoch [10/10], Loss: 0.0150


+ Model Performance

In [30]:
########### Write Your Code Here ###########

mlp_model.eval() # set the model to evaluation mode

all_predictions = [] # list to store all predicted labels
all_targets = [] # list to store all true labels

with torch.no_grad(): # disable gradient calculation for evaluation
    for batch_x, batch_y in test_loader: # iterate over the test data in batches
        # move the batch data to the device
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = mlp_model(batch_x) # compute the logits for the batch

        predictions = torch.argmax(logits, dim = 1) # get the predicted class labels (the index of the max logit)

        all_predictions.append(predictions.cpu()) # move predictions to CPU and store
        all_targets.append(batch_y.cpu()) # move true labels to CPU and store

y_predicted_cls = torch.cat(all_predictions).numpy()
# concatenate all predicted labels into a single numpy array

y_test = torch.cat(all_targets).numpy()
# concatenate all true labels into a single numpy array

accuracy = (y_predicted_cls == y_test).mean() # compute accuracy
print(f"Test Accuracy: {accuracy:.4f}") # print the test accuracy

############################################

Test Accuracy: 0.9776


In [31]:
#classification report

from sklearn.metrics import classification_report

target_names = [str(i) for i in range(10)] # class names for the classification report

print(classification_report(
    y_test,
    y_predicted_cls,
    labels = list(range(10)), # specify the order of labels in the report
    target_names = target_names, # provide class names for better readability
    digits = 4, # specify the number of decimal places for the report
    zero_division = 0 # specify how to handle zero division cases in the report
))

              precision    recall  f1-score   support

           0     0.9769    0.9908    0.9838       980
           1     0.9947    0.9850    0.9898      1135
           2     0.9739    0.9777    0.9758      1032
           3     0.9723    0.9743    0.9733      1010
           4     0.9737    0.9817    0.9777       982
           5     0.9919    0.9664    0.9790       892
           6     0.9762    0.9833    0.9797       958
           7     0.9628    0.9825    0.9726      1028
           8     0.9801    0.9610    0.9705       974
           9     0.9742    0.9713    0.9727      1009

    accuracy                         0.9776     10000
   macro avg     0.9777    0.9774    0.9775     10000
weighted avg     0.9777    0.9776    0.9776     10000

